# AC12 — Extração das disciplinas do portal do estudante (Ibmec)

Endpoint usado pelo portal `estudante.ibmec.br` para listar as turmas do período atual:

`GET https://apis.estudante.ibmec.br/rest/turmas/status?status=ATUAL`

A autenticação é por **Bearer token** (JWT do Azure AD), copiado do header `authorization` da requisição no DevTools.
O token expira em ~24h, então precisa ser recolado para reexecutar.

In [1]:
import json

import pandas as pd
import requests

URL = "https://apis.estudante.ibmec.br/rest/turmas/status"

TOKEN = "COLE_AQUI_O_BEARER"  # header authorization (sem o "Bearer ")

headers = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "authorization": f"Bearer {TOKEN}",
    "origin": "https://estudante.ibmec.br",
    "referer": "https://estudante.ibmec.br/",
    "user-agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"
    ),
}

## Requisição

In [2]:
resposta = requests.get(URL, params={"status": "ATUAL"}, headers=headers)
resposta.raise_for_status()

turmas = resposta.json()
print(resposta.status_code, "-", len(turmas), "turmas retornadas")
turmas[0]

200 - 5 turmas retornadas


{'id': 'ibmec_14736964',
 'idExterno': None,
 'marca': 'IBMEC',
 'codigoEntrega': '8001',
 'formato': 'presencial',
 'campus': 'BELO HORIZONTE - FUNCIONÁRIOS',
 'tipoCurso': 'GRADUAÇÃO',
 'educadorResponsavel': {'nome': 'ANGELICA MATOS GUIMARAES DIAS',
  'perfil': 'professor'},
 'periodoAcademico': '2026.2',
 'local': {'blocos': ['1', '1', '1', '1'],
  'salas': ['308', '308', '308', '308']},
 'horarios': [{'diaSemana': 'Qua',
   'horaInicio': '12:50:00Z',
   'horaFim': '13:45:00Z'},
  {'diaSemana': 'Qua', 'horaInicio': '13:45:00Z', 'horaFim': '14:40:00Z'},
  {'diaSemana': 'Qui', 'horaInicio': '12:50:00Z', 'horaFim': '13:45:00Z'},
  {'diaSemana': 'Qui', 'horaInicio': '13:45:00Z', 'horaFim': '14:40:00Z'}],
 'educadores': [{'nome': 'ANGELICA MATOS GUIMARAES DIAS',
   'perfil': 'professor'},
  {'nome': 'ANGELICA MATOS GUIMARAES DIAS', 'perfil': 'professor'},
  {'nome': 'ANGELICA MATOS GUIMARAES DIAS', 'perfil': 'professor'},
  {'nome': 'ANGELICA MATOS GUIMARAES DIAS', 'perfil': 'professor'

## Montando o dataframe

Cada turma traz os horários como uma lista de aulas de 55 min. A função abaixo junta as aulas
seguidas do mesmo dia em um único intervalo (`Qua 12:50-14:40`).

In [3]:
def formata_horarios(horarios):
    """Junta as aulas seguidas do mesmo dia num intervalo só."""
    por_dia = {}
    for h in horarios or []:
        dia = h["diaSemana"]
        ini, fim = h["horaInicio"][:5], h["horaFim"][:5]
        if dia in por_dia:
            por_dia[dia] = (min(por_dia[dia][0], ini), max(por_dia[dia][1], fim))
        else:
            por_dia[dia] = (ini, fim)
    return "; ".join(f"{d} {i}-{f}" for d, (i, f) in por_dia.items())


linhas = []
for t in turmas:
    local = t.get("local") or {}
    professores = sorted({e["nome"] for e in t.get("educadores") or []})
    linhas.append(
        {
            "codigo_disciplina": t["codigoDisciplina"],
            "disciplina": t["nome"],
            "periodo": t["periodoAcademico"],
            "professor_responsavel": (t.get("educadorResponsavel") or {}).get("nome"),
            "professores": ", ".join(professores),
            "horarios": formata_horarios(t.get("horarios")),
            "bloco": ", ".join(sorted(set(local.get("blocos") or []))),
            "sala": ", ".join(sorted(set(local.get("salas") or []))),
            "campus": t["campus"],
            "formato": t["formato"],
            "tipo_curso": t["tipoCurso"],
            "alunos_matriculados": t["totalAlunosMatriculados"],
            "turma_id": t["id"],
        }
    )

df = pd.DataFrame(linhas).sort_values("disciplina").reset_index(drop=True)
df

,codigo_disciplina,disciplina,periodo,professor_responsavel,professores,horarios,bloco,sala,campus,formato,tipo_curso,alunos_matriculados,turma_id
0,IBM8915,EXTRAÇÃO E PREPARAÇÃO DE DADOS,2026.2,PEDRO HENRIQUE CALAIS GUERRA,PEDRO HENRIQUE CALAIS GUERRA,Ter 12:50-14:40; Qua 10:30-12:20,1,108,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,16,ibmec_14736977
1,IBM0037,INFERÊNCIA ESTATÍSTICA,2026.2,FRANK MAGALHAES DE PINHO,FRANK MAGALHAES DE PINHO,Seg 10:30-12:20; Qui 10:30-12:20,1,313,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,48,ibmec_14736680
2,IBM1740,INOVAÇÃO E DESIGN THINKING,2026.2,TADEU MOREIRA PERONA,TADEU MOREIRA PERONA,Ter 10:30-12:20; Sex 10:30-12:20,2,206,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,21,ibmec_14736847
3,IBM0792,MÉTODOS ÁGEIS DE DESENVOLVIMENTO DE SOFTWARE,2026.2,EDES GARCIA DA COSTA FILHO,EDES GARCIA DA COSTA FILHO,Seg 12:50-14:40; Sex 12:50-14:40,1,112,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,52,ibmec_14736836
4,IBM4028,PROJETO EM CIÊNCIA DE DADOS IV,2026.2,ANGELICA MATOS GUIMARAES DIAS,ANGELICA MATOS GUIMARAES DIAS,Qua 12:50-14:40; Qui 12:50-14:40,1,308,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,8,ibmec_14736964


## Uma linha por aula (grade da semana)

In [4]:
df_horarios = (
    pd.DataFrame(turmas)[["codigoDisciplina", "nome", "horarios"]]
    .explode("horarios")
    .dropna(subset=["horarios"])
)
df_horarios = pd.concat(
    [
        df_horarios.drop(columns="horarios").reset_index(drop=True),
        pd.json_normalize(df_horarios["horarios"]).reset_index(drop=True),
    ],
    axis=1,
).rename(columns={"codigoDisciplina": "codigo_disciplina", "nome": "disciplina"})

df_horarios

,codigo_disciplina,disciplina,diaSemana,horaInicio,horaFim
0,IBM4028,PROJETO EM CIÊNCIA DE DADOS IV,Qua,12:50:00Z,13:45:00Z
1,IBM4028,PROJETO EM CIÊNCIA DE DADOS IV,Qua,13:45:00Z,14:40:00Z
2,IBM4028,PROJETO EM CIÊNCIA DE DADOS IV,Qui,12:50:00Z,13:45:00Z
3,IBM4028,PROJETO EM CIÊNCIA DE DADOS IV,Qui,13:45:00Z,14:40:00Z
4,IBM8915,EXTRAÇÃO E PREPARAÇÃO DE DADOS,Ter,12:50:00Z,13:45:00Z
5,IBM8915,EXTRAÇÃO E PREPARAÇÃO DE DADOS,Ter,13:45:00Z,14:40:00Z
6,IBM8915,EXTRAÇÃO E PREPARAÇÃO DE DADOS,Qua,10:30:00Z,11:25:00Z
7,IBM8915,EXTRAÇÃO E PREPARAÇÃO DE DADOS,Qua,11:25:00Z,12:20:00Z
8,IBM0037,INFERÊNCIA ESTATÍSTICA,Seg,10:30:00Z,11:25:00Z
9,IBM0037,INFERÊNCIA ESTATÍSTICA,Seg,11:25:00Z,12:20:00Z


> As horas vêm com sufixo `Z` na API, mas são horário local de Brasília — não são UTC de verdade.

## Salvando

In [5]:
df.to_csv("turmas-ibmec.csv", index=False, encoding="utf-8")
df_horarios.to_csv("turmas-ibmec-horarios.csv", index=False, encoding="utf-8")

with open("turmas-atual.json", "w", encoding="utf-8") as f:
    json.dump(turmas, f, ensure_ascii=False, indent=2)

print("arquivos salvos:", "turmas-ibmec.csv", "turmas-ibmec-horarios.csv", "turmas-atual.json")

arquivos salvos: turmas-ibmec.csv turmas-ibmec-horarios.csv turmas-atual.json
